In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

CREATE OR REPLACE TABLE gold.default_by_grade AS
SELECT grade,
       COUNT(*)                              AS total_loans,
       SUM(is_default)                       AS defaults,
       ROUND(SUM(is_default) / COUNT(*), 4)  AS default_rate,
       ROUND(AVG(int_rate), 2)               AS avg_interest_rate
FROM silver.loans_clean
GROUP BY grade
ORDER BY default_rate DESC;

CREATE OR REPLACE TABLE gold.default_by_purpose AS
SELECT purpose,
       COUNT(*) AS total_loans,
       ROUND(SUM(is_default) / COUNT(*), 4) AS default_rate
FROM silver.loans_clean
GROUP BY purpose
ORDER BY default_rate DESC;

CREATE OR REPLACE TABLE gold.exposure_by_state AS
SELECT addr_state,
       ROUND(SUM(loan_amnt), 0) AS total_exposure,
       COUNT(*)                 AS loan_count
FROM silver.loans_clean
GROUP BY addr_state
ORDER BY total_exposure DESC;

In [0]:
%sql
SELECT * FROM gold.default_by_grade

In [0]:
%sql 
SELECT * FROM gold.default_by_purpose

In [0]:
%sql 
SELECT * FROM gold.exposure_by_state

In [0]:
from pyspark.sql.functions import col

loans_clean = spark.table("silver.loans_clean")
assert loans_clean.filter(col("id").isNull()).count() == 0, "Null id found"
assert loans_clean.count() == loans_clean.dropDuplicates(["id"]).count(), "Duplicate ids found"